
# Visualization scrolling experiments

This notebook builds three minimal outputs to see how scroll behaves inside VS Code notebooks:

1. Baseline `PipelineWidget` iframe (expected to block notebook scroll when hovered).
2. Inline React Flow (no iframe) with `preventScrolling=False` so scrolling should pass through.
3. Iframe with a wheel-forwarding hook (tries to proxy wheel events from the iframe to the notebook).

Run in VS Code notebook view and hover each block while scrolling to compare behaviors.


In [ ]:

from hypernodes import Pipeline, node
from hypernodes.viz.js.renderer import JSRenderer
from hypernodes.viz.ui_handler import UIHandler

@node(output_name="cleaned")
def clean_text(text: str) -> str:
    return text.strip().lower()

@node(output_name="length")
def text_length(text: str) -> int:
    return len(text)

pipeline = Pipeline(nodes=[clean_text, text_length])
handler = UIHandler(pipeline, depth=2, group_inputs=True, show_output_types=True)

graph_data = handler.get_visualization_data(traverse_collapsed=True)
rf_data = JSRenderer().render(
    graph_data,
    theme="auto",
    initial_depth=2,
    pan_on_scroll=False,
    separate_outputs=False,
    show_types=True,
)



## A. Baseline iframe widget (current behavior)
Hover the canvas and try to scroll the notebook; scrolling is blocked because the React Flow app lives inside an iframe.


In [ ]:

from hypernodes.viz.visualization_widget import PipelineWidget

PipelineWidget(
    pipeline=pipeline,
    theme="auto",
    depth=2,
    separate_outputs=False,
    show_types=True,
)



## B. Inline React Flow (no iframe)
Simple inline React Flow example using CDN bundles. With `preventScrolling=False` and no iframe, mouse wheel over the canvas should scroll the notebook.


In [ ]:

from IPython.display import HTML

HTML(
    """
<div id="inline-scroll-root" style="height: 520px; border: 1px solid #d1d5db; border-radius: 10px; margin: 12px 0; position: relative; overflow: hidden;">
  <div id="inline-scroll-loading" style="padding: 12px; font-family: monospace; color: #475569;">Loading inline React Flow…</div>
</div>
<link rel="stylesheet" href="https://unpkg.com/@xyflow/react@11.10.1/dist/style.css" />
<script src="https://unpkg.com/react@18/umd/react.production.min.js"></script>
<script src="https://unpkg.com/react-dom@18/umd/react-dom.production.min.js"></script>
<script src="https://unpkg.com/@xyflow/react@11.10.1/dist/umd/reactflow.min.js"></script>
<script>
(function() {
  const rootEl = document.getElementById('inline-scroll-root');
  const loading = document.getElementById('inline-scroll-loading');
  const RF = window.ReactFlow;
  if (!RF || !window.React || !window.ReactDOM) {
    rootEl.innerHTML = '<div style="padding:12px;color:#b91c1c;">React Flow libs failed to load.</div>';
    return;
  }

  const nodes = [
    { id: 'a', position: { x: 0, y: 0 }, data: { label: 'Start' }, sourcePosition: 'bottom', targetPosition: 'top' },
    { id: 'b', position: { x: 0, y: 150 }, data: { label: 'Process' }, sourcePosition: 'bottom', targetPosition: 'top' },
    { id: 'c', position: { x: 0, y: 300 }, data: { label: 'Done' }, sourcePosition: 'bottom', targetPosition: 'top' },
  ];
  const edges = [
    { id: 'e1', source: 'a', target: 'b' },
    { id: 'e2', source: 'b', target: 'c' },
  ];

  function InlineFlow() {
    return React.createElement(RF.ReactFlow, {
      nodes,
      edges,
      fitView: true,
      panOnScroll: false,
      zoomOnScroll: false,
      preventScrolling: false,
      nodesDraggable: false,
      nodesConnectable: false,
      elementsSelectable: false,
      proOptions: { hideAttribution: true },
      style: { width: '100%', height: '100%', background: 'linear-gradient(180deg,#f8fafc 0%,#eef2ff 100%)' },
    }, [
      React.createElement(RF.Background, { key: 'bg', color: '#cbd5e1', gap: 24 }),
      React.createElement(RF.Controls, { key: 'ctrl' }),
    ]);
  }

  ReactDOM.createRoot(rootEl).render(React.createElement(InlineFlow));
  if (loading) loading.remove();
})();
</script>
    """
)



## C. Iframe with wheel forwarding (experimental)
This builds the standard iframe HTML via `generate_widget_html` and attaches a parent-side wheel listener to proxy scroll to the notebook. Depending on VS Code’s webview sandboxing, this may or may not take effect.


In [ ]:

import html as py_html
from IPython.display import HTML
from hypernodes.viz.js.html_generator import generate_widget_html

iframe_doc = generate_widget_html(rf_data)
escaped = py_html.escape(iframe_doc, quote=True)

HTML(
    f"""
<div style='border: 1px solid #d1d5db; border-radius: 10px; margin: 12px 0; overflow: hidden; position: relative;'>
  <div style='padding: 8px; font-family: monospace; font-size: 12px; color: #475569;'>Wheel forwarding experiment</div>
  <iframe id='wheel-forward-iframe' srcdoc="{escaped}" sandbox='allow-scripts allow-same-origin allow-popups allow-forms' style='width: 100%; height: 520px; border: none;'></iframe>
</div>
<script>
(function() {
  const iframe = document.getElementById('wheel-forward-iframe');
  const forward = (evt) => {
    const delta = evt.deltaY || 0;
    window.scrollBy({ top: delta, left: 0, behavior: 'auto' });
    evt.preventDefault();
  };
  iframe.addEventListener('load', () => {
    try {
      const w = iframe.contentWindow;
      if (!w) return;
      w.addEventListener('wheel', forward, { capture: true, passive: false });
    } catch (err) {
      console.warn('wheel forwarding not attached', err);
    }
  });
})();
</script>
    """
)
